In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

import json
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from matplotlib.patches import Patch

import holidays
import time
from dateutil.tz import *

from sklearn.neural_network import  MLPRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.inspection import permutation_importance

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

In [2]:
start_time = time.monotonic()

In [3]:
import random

random.seed(42)
np.random.seed(42)

In [4]:
import calendar

def findDay(date):
	year, month, day = (int(i) for i in date.split('-')) 
	return calendar.weekday(year, month, day)

In [11]:
def split_and_preprocess(df):
        clients_data = []
        clients_test = []
        clients_scalers = []
        clients_imputers = []  # optional, for reproducibility

        clients_test_original = []  # (X_test_original_i, Y_test_original_i)
        clients_val_original = []  # (X_val_original_i, Y_val_original_i)

        #for sub_df in df:
        X, Y = make_XY(df)

        X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, shuffle=False)
        X_train2, X_val, Y_train2, Y_val = train_test_split(X_train, Y_train, test_size=0.2, shuffle=False)

        # 1) Impute (fit only on training)
        imputer_X = SimpleImputer(strategy="median")
        X_train_imp = imputer_X.fit_transform(X_train)
        X_val_imp   = imputer_X.transform(X_val)
        X_test_imp  = imputer_X.transform(X_test)

        # impute first if needed, then scale (as you already do)
        scaler_X = StandardScaler()
        X_train_scaled = scaler_X.fit_transform(X_train_imp)
        X_val_scaled   = scaler_X.transform(X_val_imp)
        X_test_scaled  = scaler_X.transform(X_test_imp)

        scaler_Y = StandardScaler()
        Y_train_scaled = scaler_Y.fit_transform(Y_train.reshape(-1,1)).ravel()
        Y_val_scaled   = scaler_Y.transform(Y_val.reshape(-1,1)).ravel()
        Y_test_scaled  = scaler_Y.transform(Y_test.reshape(-1,1)).ravel()

        # save scaled for training/testing
        clients_data.append((X_train_scaled, Y_train_scaled))
        clients_test.append((X_test_scaled, Y_test_scaled))
        clients_scalers.append((scaler_X, scaler_Y))
        clients_imputers.append(imputer_X)

        # save ORIGINAL (per client)
        X_test_original_i = scaler_X.inverse_transform(X_test_scaled)
        Y_test_original_i = scaler_Y.inverse_transform(Y_test_scaled.reshape(-1,1)).ravel()
        clients_test_original.append((X_test_original_i, Y_test_original_i))

        X_val_original_i = scaler_X.inverse_transform(X_val_scaled)
        Y_val_original_i = scaler_Y.inverse_transform(Y_val_scaled.reshape(-1,1)).ravel()
        clients_val_original.append((X_val_original_i, Y_val_original_i))
        return (clients_data, clients_test, clients_scalers, clients_test_original, clients_val_original)

In [ ]:
def train_MLP_client_proc(client_data_tuple): 
    model = MLPRegressor(hidden_layer_sizes=(32,),
                        activation="relu",
                        solver="adam",
                        alpha=1e-3,
                        learning_rate="adaptive",
                        learning_rate_init=1e-3,
                        max_iter=1000,
                        early_stopping=True,
                        validation_fraction=0.1,
                        n_iter_no_change=20,
                        random_state=42).fit(
                                            client_data_tuple[0], 
                                            client_data_tuple[1]
                                            )
    return model
def train_SGD_client_proc(client_data_tuple): 
    model = SGDRegressor(penalty="l2",
                        alpha=1e-4,
                        learning_rate="adaptive",
                        eta0=1e-3,
                        max_iter=2000,
                        tol=1e-4,
                        early_stopping=True,
                        validation_fraction=0.1,
                        n_iter_no_change=20,
                        random_state=42
                    ).fit(client_data_tuple[0],client_data_tuple[1])
    return model

In [13]:
def train_MLP_model(clients_data):
    num_of_clients = len(clients_data)
    print(f"Training {num_of_clients} clients in parallel...")
    max_workers = num_of_clients

    # Each client trains its own MLP model
    client_models = []
    # Train all clients in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [
                executor.submit(train_MLP_client_proc, clients_data[j])
                for j in range(num_of_clients)
            ]
            for f in as_completed(futures):
                client_models.append(f.result())
    return client_models

def train_SGD_model(clients_data):
    num_of_clients = len(clients_data)
    print(f"Training {num_of_clients} shards in parallel...")
    max_workers = num_of_clients

    # Each client trains its own SGD model
    client_models = []
    # Train all clients in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [
                executor.submit(train_SGD_client_proc, clients_data[j])
                for j in range(num_of_clients)
            ]
            for f in as_completed(futures):
                client_models.append(f.result())
    return client_models

In [14]:
# Extract weights from a trained client model
# MLPRegressor stores learned layer weights in coefs_ and biases in intercepts_.
def get_MLP_model_weights(model):
    return {
        "coefs": [w.copy() for w in model.coefs_],
        "intercepts": [b.copy() for b in model.intercepts_]
    }

# Extract weights from a trained client model
# SGDRegressor stores learned layer weights in coefs_ and biases in intercepts_[0].
def get_SGD_model_weights(model):
    return {
        "coef": model.coef_.copy(),
        "intercept": np.array([model.intercept_[0]])
    }

In [15]:
# Put averaged weights into a server/global model
def average_MLP_model_weights(client_weight_list):
    n_clients = len(client_weight_list)
    print("number of clients ", n_clients)
    
    avg_coefs = []
    avg_intercepts = []

    n_layers = len(client_weight_list[0]["coefs"])

    for layer_idx in range(n_layers):
        layer_weights = [cw["coefs"][layer_idx] for cw in client_weight_list]
        layer_biases  = [cw["intercepts"][layer_idx] for cw in client_weight_list]

        avg_coefs.append(np.mean(layer_weights, axis=0))
        avg_intercepts.append(np.mean(layer_biases, axis=0))

    return {
        "coefs": avg_coefs,
        "intercepts": avg_intercepts
    }

# Put averaged weights into a server/global model
def average_SGD_model_weights(client_weights):
    avg_coef = np.mean([w["coef"] for w in client_weights], axis=0)
    avg_intercept = np.mean([w["intercept"] for w in client_weights], axis=0)
    return {"coef": avg_coef, "intercept": avg_intercept}

In [16]:
# Put averaged weights into a server/global model
def build_global_MLP_model_like(client_models, X_sample, y_sample):
    """
    Create a model with same architecture as client_models[0]
    and initialize internal structures via a dummy fit.
    """
    base = client_models[0]

    global_model = MLPRegressor(
        hidden_layer_sizes=base.hidden_layer_sizes,
        activation=base.activation,
        solver=base.solver,
        alpha=base.alpha,
        batch_size=base.batch_size,
        learning_rate=base.learning_rate,
        learning_rate_init=base.learning_rate_init,
        max_iter=1,
        random_state=base.random_state
    )

    # fit to initialize shapes
    global_model.fit(X_sample[:2], y_sample[:2])

    return global_model

# Put averaged weights into a server/global model
def build_global_SGD_model_like(client_models, X_sample, y_sample):
    """
    Create a model with same architecture as client_models[0]
    and initialize internal structures via a dummy fit.
    """
    base = client_models[0]

    global_model = SGDRegressor(
        random_state=base.random_state
    )

    # fit to initialize shapes
    global_model.fit(X_sample[:2], y_sample[:2])

    return global_model

In [17]:
# Assign averaged weights to the global model
def set_MLP_model_weights(model, weights):
    model.coefs_ = [w.copy() for w in weights["coefs"]]
    model.intercepts_ = [b.copy() for b in weights["intercepts"]]
    return model

# Assign averaged weights to the global model
def set_SGD_model_weights(model, weights):
    model.coef_ = weights["coef"].copy()
    model.intercept_ = weights["intercept"].copy()
    return model

In [18]:
# Assign averaged weights to the global model
def fedavg_mlp(client_models, X_sample, y_sample):
    # extract
    #print("Extracting client model weights...")
    client_weight_list = [get_MLP_model_weights(m) for m in client_models]

    # average
    avg_weights = average_MLP_model_weights(client_weight_list)

    # build server/global model
    global_model = build_global_MLP_model_like(client_models, X_sample, y_sample)

    # load averaged weights
    global_model = set_MLP_model_weights(global_model, avg_weights)

    return global_model

# Assign averaged weights to the global model
def fedavg_sgd(client_models, X_sample, y_sample):
    # extract
    #print("Extracting client model weights...")
    client_weight_list = [get_SGD_model_weights(m) for m in client_models]

    # average
    avg_weights = average_SGD_model_weights(client_weight_list)

    # build server/global model
    global_model = build_global_SGD_model_like(client_models, X_sample, y_sample)

    # load averaged weights
    global_model = set_SGD_model_weights(global_model, avg_weights)

    return global_model

In [ ]:
def infer_with_global_MLP_model(global_model, clients_test, clients_scalers):
    prediction_per_shard = []
    prediction_time_per_shard = []
    loss_curves = []
    ytest_per_shard = []

    for i, (X_test_i, Y_test_i_scaled) in enumerate(clients_test):
        start = time.perf_counter()
        _, scaler_Y = clients_scalers[i]

        pred_i_scaled = global_model.predict(X_test_i).reshape(-1, 1)
        loss_curve_i = global_model.loss_curve_ 
        pred_i = scaler_Y.inverse_transform(pred_i_scaled).reshape(-1)
        pred_i = np.clip(pred_i, 0.0, None)

        # also reconstruct Y_test in original units if you want plots/metrics
        y_i = scaler_Y.inverse_transform(Y_test_i_scaled.reshape(-1, 1)).reshape(-1)

        prediction_per_shard.append(pred_i)
        loss_curves.append(loss_curve_i)
        ytest_per_shard.append(y_i)
        pred_time_s = time.perf_counter() - start
        prediction_time_per_shard.append(pred_time_s)

    return prediction_per_shard, prediction_time_per_shard, loss_curves

def infer_with_global_SGD_model(global_model, clients_test, clients_scalers):
    prediction_per_shard = []
    prediction_time_per_shard = []
    loss_curves = []
    ytest_per_shard = []

    for i, (X_test_i, Y_test_i_scaled) in enumerate(clients_test):
        start = time.perf_counter()
        _, scaler_Y = clients_scalers[i]

        pred_i_scaled = global_model.predict(X_test_i).reshape(-1, 1)
        pred_i = scaler_Y.inverse_transform(pred_i_scaled).reshape(-1)
        pred_i = np.clip(pred_i, 0.0, None)

        # also reconstruct Y_test in original units if you want plots/metrics
        y_i = scaler_Y.inverse_transform(Y_test_i_scaled.reshape(-1, 1)).reshape(-1)

        prediction_per_shard.append(pred_i)
        ytest_per_shard.append(y_i)
        pred_time_s = time.perf_counter() - start
        prediction_time_per_shard.append(pred_time_s)

    return prediction_per_shard, prediction_time_per_shard 

In [ ]:
def locally_infer_from_MLP_model(client_models, clients_test, clients_scalers):
    prediction_per_shard = []
    prediction_time_per_shard = []
    loss_curves = []
    ytest_per_shard = []

    for i, (X_test_i, Y_test_i_scaled) in enumerate(clients_test):
        start = time.perf_counter()
        _, scaler_Y = clients_scalers[i]

        pred_i_scaled = client_models[i].predict(X_test_i).reshape(-1, 1)
        loss_curve_i = client_models[i].loss_curve_
        pred_i = scaler_Y.inverse_transform(pred_i_scaled).reshape(-1)
        pred_i = np.clip(pred_i, 0.0, None)

        # also reconstruct Y_test in original units if you want plots/metrics
        y_i = scaler_Y.inverse_transform(Y_test_i_scaled.reshape(-1, 1)).reshape(-1)

        prediction_per_shard.append(pred_i)
        loss_curves.append(loss_curve_i)
        ytest_per_shard.append(y_i)
        pred_time_s = time.perf_counter() - start
        prediction_time_per_shard.append(pred_time_s)

    return prediction_per_shard, prediction_time_per_shard, loss_curves

def locally_infer_from_SGD_model(client_models, clients_test, clients_scalers):
    prediction_per_shard = []
    prediction_time_per_shard = []
    loss_curves = []
    ytest_per_shard = []

    for i, (X_test_i, Y_test_i_scaled) in enumerate(clients_test):
        start = time.perf_counter()
        _, scaler_Y = clients_scalers[i]

        pred_i_scaled = client_models[i].predict(X_test_i).reshape(-1, 1)
        pred_i = scaler_Y.inverse_transform(pred_i_scaled).reshape(-1)
        pred_i = np.clip(pred_i, 0.0, None)

        # also reconstruct Y_test in original units if you want plots/metrics
        y_i = scaler_Y.inverse_transform(Y_test_i_scaled.reshape(-1, 1)).reshape(-1)

        prediction_per_shard.append(pred_i)
        ytest_per_shard.append(y_i)
        pred_time_s = time.perf_counter() - start
        prediction_time_per_shard.append(pred_time_s)

    return prediction_per_shard, prediction_time_per_shard

In [21]:
def compute_error(Y_test_original_all, pred):
    mae = np.round(mean_absolute_error(Y_test_original_all, pred),4)
    mse = mean_squared_error(Y_test_original_all, pred)
    rmse = np.round(mse ** 0.5, 4)
    r2 = np.round(r2_score(Y_test_original_all, pred),4)
    return mae, mse, rmse, r2

In [ ]:
# -----------------------------
# Helper: build similar shards
# -----------------------------

def _cluster_shards(num_shards):
    """
    select households and split into `num_shards` groups.
    Returns: List[List[hh_id]]
    """

# -----------------------------
# Helper: build random shards
# -----------------------------
def _random_shards(hh_ids, num_shards, rng):
    """
    Randomly permute households and split into `num_shards` groups
    as evenly as possible (for your cases, sizes are exact).
    Returns: List[List[hh_id]]
    """
    print(f"Creating {num_shards} random shards from {len(hh_ids)} households...")
    hh_ids = list(hh_ids)
    perm = rng.permutation(hh_ids).tolist()
    return [perm[i::num_shards] for i in range(num_shards)]


# -----------------------------
# Helper: aggregate data per shard
# -----------------------------
def _aggregate_households(datasets_by_hh, shard_hh_ids):
    """
    Given mapping {hh_id: df_or_data}, aggregate the households listed in shard_hh_ids
    into one shard dataset.
    You will implement actual aggregation (concat, sort by time, feature engineering, etc.).
    """
    shard_frames = []

    for hid in shard_hh_ids:
        df = datasets_by_hh[hid].copy()
        df["hh_id"] = hid
        shard_frames.append(df)

    shard_df = pd.concat(shard_frames, axis=0, ignore_index=True)

    return shard_df


# -----------------------------
# Helper: MLP train + aggregate + infer + error
# -----------------------------
def _MLP_train_aggregate_infer_error(shard_datasets, shard_hh_ids, type, features_cols):

    all_clients_data = []
    all_clients_test = []
    all_clients_scalers = []
    all_clients_test_original = []
    all_clients_val_original = []

    # process each shard separately
    for shard_dataset in shard_datasets:

        clients_data, clients_test, clients_scalers, clients_test_original, clients_val_original = split_and_preprocess(shard_dataset)

        all_clients_data.extend(clients_data)
        all_clients_test.extend(clients_test)
        all_clients_scalers.extend(clients_scalers)
        all_clients_test_original.extend(clients_test_original)
        all_clients_val_original.extend(clients_val_original)


    if type == "Federated":
        # train models
        client_models = train_MLP_model(all_clients_data)
        
        # aggregate models
        global_model = fedavg_mlp(client_models, all_clients_data[0][0], all_clients_data[0][1])

        # predict with global model
        start = time.perf_counter()
        prediction_per_shard, prediction_time_per_shard, loss_curves = infer_with_global_MLP_model(
            global_model , all_clients_test, all_clients_scalers
        )
        prediction_time_total = time.perf_counter() - start
        mse_num_shards = []
        r2_scores = []
        for j, model in enumerate(client_models):
            X_val, Y_val = all_clients_val_original[j]
            Y_pred = model.predict(X_val)
            mse_num_shards.append(mean_squared_error(Y_val, Y_pred))
            r2_scores.append(r2_score(Y_val, Y_pred))
        print("Federated - MSE: ", len(mse_num_shards), " ", (mse_num_shards))
        print("Federated - R2: ", len(r2_scores), " ", (r2_scores))
    elif type == "Local":
        # train models
        client_models = train_MLP_model(all_clients_data)
        
        # predict with global model
        start = time.perf_counter()
        prediction_per_shard, prediction_time_per_shard, loss_curves = locally_infer_from_MLP_model(
            client_models, all_clients_test, all_clients_scalers
        )
        prediction_time_total = time.perf_counter() - start
        
        r2_scores = []
        mse_num_shards = []
        for j, model in enumerate(client_models):
            X_val, Y_val = all_clients_val_original[j]
            Y_pred = model.predict(X_val)
            mse_num_shards.append(mean_squared_error(Y_val, Y_pred))
            r2_scores.append(r2_score(Y_val, Y_pred))
        print("Local - MSE: ", len(mse_num_shards), " ", (mse_num_shards))
        print("Local - R2: ", len(r2_scores), " ", (r2_scores))
        
    elif type == "Centralized":
        # train model on all data
        X_all = np.concatenate([cd[0] for cd in all_clients_data])
        Y_all = np.concatenate([cd[1] for cd in all_clients_data])
        global_model = MLPRegressor(solver = 'adam', 
                                    max_iter = 10000,
                                    random_state=42).fit(X_all, Y_all)
        print("Trained centralized MLP model.")
        
        # predict with global model
        start = time.perf_counter()
        prediction_per_shard, prediction_time_per_shard, loss_curves = infer_with_global_MLP_model(
            global_model , all_clients_test, all_clients_scalers
        )
        prediction_time_total = time.perf_counter() - start

    mae, mse, rmse, r2 = [], [], [], []
    for i, (Xo, Yo) in enumerate(all_clients_test_original):
        mae_val, mse_val, rmse_val, r2_val = compute_error(
                                                            Yo, 
                                                            prediction_per_shard[i]
                                                            )
        mae.append(mae_val)
        mse.append(mse_val)
        rmse.append(rmse_val)
        r2.append(r2_val)
    return {
        "shard_hh_ids": shard_hh_ids,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
        "prediction_per_shard": ((prediction_per_shard)),
        "prediction_time_per_shard": np.round(np.array(prediction_time_per_shard), 4),
        "prediction_time_total": np.round(float(prediction_time_total), 4),
        "clients_test_original": all_clients_test_original
    }

# -----------------------------
# Helper: SGD train + aggregate + infer + error
# -----------------------------
def _SGD_train_aggregate_infer_error(shard_datasets, shard_hh_ids, type, features_cols):

    all_clients_data = []
    all_clients_test = []
    all_clients_scalers = []
    all_clients_test_original = []
    all_clients_val_original = []

    # process each shard separately
    for shard_dataset in shard_datasets:

        clients_data, clients_test, clients_scalers, clients_test_original, clients_val_original = split_and_preprocess(shard_dataset)

        all_clients_data.extend(clients_data)
        all_clients_test.extend(clients_test)
        all_clients_scalers.extend(clients_scalers)
        all_clients_test_original.extend(clients_test_original)
        all_clients_val_original.extend(clients_val_original)

    if type == "Federated":
        # train models
        client_models = train_SGD_model(all_clients_data)
        
        # aggregate models
        global_model = fedavg_sgd(client_models, all_clients_data[0][0], all_clients_data[0][1])

        # predict with global model
        start = time.perf_counter()
        prediction_per_shard, prediction_time_per_shard = infer_with_global_SGD_model(
            global_model , all_clients_test, all_clients_scalers
        )
        prediction_time_total = time.perf_counter() - start
        mse_num_shards = []
        r2_scores = []
        for j, model in enumerate(client_models):
            X_val, Y_val = all_clients_val_original[j]
            Y_pred = model.predict(X_val)
            mse_num_shards.append(mean_squared_error(Y_val, Y_pred))
            r2_scores.append(r2_score(Y_val, Y_pred))
        print("Federated - MSE: ", len(mse_num_shards), " ", (mse_num_shards))
        print("Federated - R2: ", len(r2_scores), " ", (r2_scores))
        
    elif type == "Local":
        # train models
        client_models = train_SGD_model(all_clients_data)
        
        # predict with global model
        start = time.perf_counter()
        prediction_per_shard, prediction_time_per_shard = locally_infer_from_SGD_model(
            client_models, all_clients_test, all_clients_scalers
        )
        prediction_time_total = time.perf_counter() - start
        r2_scores = []
        mse_num_shards = []
        for j, model in enumerate(client_models):
            X_val, Y_val = all_clients_val_original[j]
            Y_pred = model.predict(X_val)
            mse_num_shards.append(mean_squared_error(Y_val, Y_pred))
            r2_scores.append(r2_score(Y_val, Y_pred))
        print("Local - MSE: ", len(mse_num_shards), " ", (mse_num_shards))
        print("Local - R2: ", len(r2_scores), " ", (r2_scores))

    elif type == "Centralized":
        # train model on all data
        X_all = np.concatenate([cd[0] for cd in all_clients_data])
        Y_all = np.concatenate([cd[1] for cd in all_clients_data])
        global_model = SGDRegressor(random_state=42).fit(X_all, Y_all)
        print("Trained centralized SGD model.")

        # predict with global model
        start = time.perf_counter()
        prediction_per_shard, prediction_time_per_shard = infer_with_global_SGD_model(
            global_model , all_clients_test, all_clients_scalers
        )
        prediction_time_total = time.perf_counter() - start
        r2_scores = []
        mse_num_shards = [] 
        for client_val_data in all_clients_val_original:
            X_val, Y_val = client_val_data
            Y_pred = global_model.predict(X_val)
            mse_num_shards.append(mean_squared_error(Y_val, Y_pred))
            r2_scores.append(r2_score(Y_val, Y_pred))
        print("Centralized - MSE: ", len(mse_num_shards), " ", (mse_num_shards))
        print("Centralized - R2: ", len(r2_scores), " ", (r2_scores))

    mae, mse, rmse, r2 = [], [], [], []
    for i, (Xo, Yo) in enumerate(all_clients_test_original):
        mae_val, mse_val, rmse_val, r2_val = compute_error(Yo,prediction_per_shard[i])
        mae.append(mae_val)
        mse.append(mse_val)
        rmse.append(rmse_val)
        r2.append(r2_val)
    
    return {
        "shard_hh_ids": shard_hh_ids,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
        "prediction_per_shard": ((prediction_per_shard)),
        "prediction_time_per_shard": np.round(np.array(prediction_time_per_shard), 4),
        "prediction_time_total": np.round(float(prediction_time_total), 4),
        "clients_test_original": all_clients_test_original
    }

# -----------------------------
# Scenario runners
# -----------------------------
def Federated_shards(datasets_by_hh, model, run_id, rng, num_shards, keep_cols=None):
    """
    num_shards -> num_shards HHs isolated (12/num_shards HH per shard)
    """
    start = time.perf_counter()
    hh_ids = list(datasets_by_hh.keys())
    if rng is not None:
        shards = _random_shards(hh_ids, num_shards=num_shards, rng=rng)  # will produce 12 groups of size 1
    elif rng is None:
        shards = _cluster_shards(num_shards=num_shards)  # will produce 12 groups of size 1
    print(f"Shards (HH IDs): {shards}")
    shard_datasets = []
    shard_hh_ids_list = []
    errors = []
    rows = []  # list of DataFrames, concat once at the end
    prediction_time_total = []
    prediction_time_per_shard = []
    for shard_hh_ids in shards:
        shard_dataset = _aggregate_households(datasets_by_hh, shard_hh_ids)
        shard_hh_ids_list.append(shard_hh_ids)
        shard_datasets.append(shard_dataset)

    if keep_cols is None:
        features_cols = ["air1_lag_1h",  "furnace1_lag_1h", "usage_lag_1h", "Month","Weekday", "Type_of_day","Hour"]
    else:
        features_cols = [f"x{c}" for c in keep_cols]

    if model == "MLP":
        result = _MLP_train_aggregate_infer_error(shard_datasets, shard_hh_ids_list, type="Federated", features_cols=features_cols)
    elif model == "SGD":
        result = _SGD_train_aggregate_infer_error(shard_datasets, shard_hh_ids_list, type="Federated", features_cols=features_cols)
    prediction_time_per_shard.append(result["prediction_time_per_shard"])
    prediction_time_total.append(result["prediction_time_total"])
    errors.append({
            "mae": result["mae"],
            "mse": result["mse"],
            "rmse": result["rmse"],
            "r2": result["r2"]
        })

    for i, shard_hh_ids in enumerate(shard_hh_ids_list):
        Xo, Yo = result["clients_test_original"][i]
        prediction_per_shard = result["prediction_per_shard"][i]

        Xo = np.asarray(Xo)
        Yo = np.asarray(Yo).reshape(-1)
        prediction_per_shard = np.asarray(prediction_per_shard).reshape(-1)

        if keep_cols is None:
            cols = ["air1_lag_1h",  "furnace1_lag_1h", "usage_lag_1h", "Month","Weekday", "Type_of_day","Hour"]
            df_tmp = pd.DataFrame(Xo, columns=cols)
        else:
            df_tmp = pd.DataFrame(Xo[:, keep_cols], columns=[f"x{c}" for c in keep_cols])

        df_tmp["hh_id"] = str(shard_hh_ids)
        df_tmp["Actual"] = Yo
        df_tmp["Predicted"] = prediction_per_shard

        rows.append(df_tmp)

    pred_test_df = pd.concat(rows, ignore_index=True)
    completion_time_s = time.perf_counter() - start
    return errors, pred_test_df, prediction_time_per_shard, prediction_time_total, np.round(float(completion_time_s), 4) #, loss_curves_shards 

def Centralized(datasets_by_hh, model, run_id, rng=None, keep_cols=None):
    """
    Centralized -> all 12 HHs aggregated into one dataset, one model.
    rng not used (kept for interface symmetry).
    """
    start = time.perf_counter()
    hh_ids = list(datasets_by_hh.keys())
    shard_dataset = _aggregate_households(datasets_by_hh, hh_ids)
    
    errors = []
    rows = []  # list of DataFrames, concat once at the end
    prediction_time_per_shard = []
    prediction_time_total = []

    if keep_cols is None:
        features_cols = ["air1_lag_1h",  "furnace1_lag_1h", "usage_lag_1h", "Month","Weekday", "Type_of_day","Hour"]
    else:
        features_cols = [f"x{c}" for c in keep_cols]

    if model == "MLP":
        result = _MLP_train_aggregate_infer_error([shard_dataset], [hh_ids], type="Centralized", features_cols=features_cols)
    elif model == "SGD":
        result = _SGD_train_aggregate_infer_error([shard_dataset], [hh_ids], type="Centralized", features_cols=features_cols)

    prediction_time_per_shard.append(result["prediction_time_per_shard"])
    prediction_time_total.append(result["prediction_time_total"])
    errors.append({
            "mae": result["mae"],
            "mse": result["mse"],
            "rmse": result["rmse"],
            "r2": result["r2"]
        })

    Xo, Yo = result["clients_test_original"][0]
    prediction_per_shard = result["prediction_per_shard"][0]

    Xo = np.asarray(Xo)
    Yo = np.asarray(Yo).reshape(-1)
    prediction_per_shard = np.asarray(prediction_per_shard).reshape(-1)

    if keep_cols is None:
        cols = ["air1_lag_1h",  "furnace1_lag_1h", "usage_lag_1h", "Month","Weekday", "Type_of_day","Hour"]
        df_tmp = pd.DataFrame(Xo, columns=cols)
    else:
        df_tmp = pd.DataFrame(Xo[:, keep_cols], columns=[f"x{c}" for c in keep_cols])

    df_tmp["hh_id"] = "Centralized"
    df_tmp["Actual"] = Yo
    df_tmp["Predicted"] = prediction_per_shard

    rows.append(df_tmp)
    pred_test_df = pd.concat(rows, ignore_index=True)
    
    completion_time_s = time.perf_counter() - start
    print(f"Completed in {completion_time_s:.4f} seconds")
    return errors, pred_test_df, prediction_time_per_shard, prediction_time_total, np.round(float(completion_time_s), 4) #, result["loss_curves"]
    
def Localized(datasets_by_hh, model, run_id, rng=None, keep_cols=None):
    """
    Localized -> each HH is its own shard, 12 models, no aggregation.
    rng not used (kept for interface symmetry).
    """
    start = time.perf_counter()
    hh_ids = list(datasets_by_hh.keys())
    
    errors = []
    rows = []  # list of DataFrames, concat once at the end
    prediction_time_per_shard = []
    prediction_time_total = []

    if keep_cols is None:
        features_cols = ["air1_lag_1h",  "furnace1_lag_1h", "usage_lag_1h", "Month","Weekday", "Type_of_day","Hour"]
    else:
        features_cols = [f"x{c}" for c in keep_cols]

    for hh_id in hh_ids:
        shard_dataset = _aggregate_households(datasets_by_hh, [hh_id])
        if model == "MLP":
            result = _MLP_train_aggregate_infer_error([shard_dataset], [[hh_id]], type="Local", features_cols=features_cols)
        elif model == "SGD":
            result = _SGD_train_aggregate_infer_error([shard_dataset], [[hh_id]], type="Local", features_cols=features_cols)

        prediction_time_per_shard.append(result["prediction_time_per_shard"])
        prediction_time_total.append(result["prediction_time_total"])
        errors.append({
                "mae": result["mae"],
                "mse": result["mse"],
                "rmse": result["rmse"],
                "r2": result["r2"]
            })

        Xo, Yo = result["clients_test_original"][0]
        prediction_per_shard = result["prediction_per_shard"][0]

        Xo = np.asarray(Xo)
        Yo = np.asarray(Yo).reshape(-1)
        prediction_per_shard = np.asarray(prediction_per_shard).reshape(-1)

        if keep_cols is None:
            cols = ["air1_lag_1h",  "furnace1_lag_1h", "usage_lag_1h", "Month","Weekday", "Type_of_day","Hour"]
            df_tmp = pd.DataFrame(Xo, columns=cols)
        else:
            df_tmp = pd.DataFrame(Xo[:, keep_cols], columns=[f"x{c}" for c in keep_cols])

        df_tmp["hh_id"] = str(hh_id)
        df_tmp["Actual"] = Yo
        df_tmp["Predicted"] = prediction_per_shard

        rows.append(df_tmp)

    pred_test_df = pd.concat(rows, ignore_index=True)
    
    completion_time_s = time.perf_counter() - start
    print(f"Completed in {completion_time_s:.4f} seconds")
    return errors, pred_test_df, prediction_time_per_shard, prediction_time_total, np.round(float(completion_time_s), 4) #, result["loss_curves"]


# -----------------------------
# Experiment: repeat n times
# -----------------------------
def run_experiments(datasets_by_hh, model, seed, repeats):
    """
    Repeats each scenario n times:
      - random HH permutation
      - shard assignment
      - aggregation
      - train
      - aggregate
      - infer
      - error

    Returns a dict of results you can post-process (mean/std, boxplots, etc.).
    """

    if seed is not None:
        rng = np.random.default_rng(seed)
    elif seed is None:
        rng = None

    results = {
        "Cent.": {},
        "Local.": {},
        "Fed_2shards": {},
        "Fed_3shards": {},
        "Fed_4shards": {},
        "Fed_6shards": {},
        "Fed_12shards": {}
    }


    for run_id in range(repeats):
        results[f"Cent."][run_id] = Centralized(
                datasets_by_hh, model, run_id, rng=None, keep_cols=None
            )
        results[f"Local."][run_id] = Localized(
                datasets_by_hh, model, run_id, rng=None, keep_cols=None
            )
        for i in [2,3,4,6,12]:
            results[f"Fed_{i}shards"][run_id] = Federated_shards(
                datasets_by_hh, model, run_id, rng, i, keep_cols=None
            )
    print(f"All experiments of {model} with seed {seed} is completed.")
    return results

In [ ]:
datasets_by_hh = {
    }

In [24]:
results_rand_MLP = run_experiments(datasets_by_hh, "MLP", seed=0, repeats=2)

Trained centralized MLP model.
Completed in 436.4187 seconds
Training 1 clients in parallel...
Local - MSE:  1   [16.235545049309135]
Local - R2:  1   [-325.8236463650852]
Training 1 clients in parallel...
Local - MSE:  1   [16.93298084027395]
Local - R2:  1   [-63.025727977572856]
Training 1 clients in parallel...
Local - MSE:  1   [6.666106794898692]
Local - R2:  1   [-54.712842629385534]
Training 1 clients in parallel...
Local - MSE:  1   [19.049644099242723]
Local - R2:  1   [-322.6422353340756]
Training 1 clients in parallel...
Local - MSE:  1   [3.874270892896307]
Local - R2:  1   [-382.2279824033895]
Training 1 clients in parallel...
Local - MSE:  1   [104.44034780636284]
Local - R2:  1   [-457.7546078721835]
Training 1 clients in parallel...
Local - MSE:  1   [6.457226989911679]
Local - R2:  1   [-17.986162447307173]
Training 1 clients in parallel...
Local - MSE:  1   [23.03077734728223]
Local - R2:  1   [-66.83526307337551]
Training 1 clients in parallel...
Local - MSE:  1   [

In [25]:
print(results_rand_MLP)

{'Cent.': {0: ([{'mae': [np.float64(0.7883)], 'mse': [7.0560028282552585], 'rmse': [np.float64(2.6563)], 'r2': [np.float64(-2.6465)]}],          air1_lag_1h  furnace1_lag_1h  usage_lag_1h  Month  Weekday  \
0              0.000            0.679         1.075   10.0      0.0   
1              0.000            0.678         1.084   10.0      0.0   
2              0.000            0.678         1.080   10.0      0.0   
3              0.000            0.678         1.077   10.0      0.0   
4              0.000            0.678         1.071   10.0      0.0   
...              ...              ...           ...    ...      ...   
1256783        0.041            0.012         0.590    2.0      3.0   
1256784        0.041            0.012         0.582    2.0      3.0   
1256785        0.041            0.012         0.576    2.0      3.0   
1256786        0.041            0.012         0.577    2.0      3.0   
1256787        0.041            0.012         0.582    2.0      3.0   

         Ty

In [ ]:
results_sim_MLP = run_experiments(datasets_by_hh, "MLP", seed=None, repeats=2)

Trained centralized MLP model.
Completed in 420.7810 seconds
Training 1 clients in parallel...
Local - MSE:  1   [16.235545049309135]
Local - R2:  1   [-325.8236463650852]
Training 1 clients in parallel...
Local - MSE:  1   [16.93298084027395]
Local - R2:  1   [-63.025727977572856]
Training 1 clients in parallel...
Local - MSE:  1   [6.666106794898692]
Local - R2:  1   [-54.712842629385534]
Training 1 clients in parallel...
Local - MSE:  1   [19.049644099242723]
Local - R2:  1   [-322.6422353340756]
Training 1 clients in parallel...
Local - MSE:  1   [3.874270892896307]
Local - R2:  1   [-382.2279824033895]
Training 1 clients in parallel...
Local - MSE:  1   [104.44034780636284]
Local - R2:  1   [-457.7546078721835]
Training 1 clients in parallel...
Local - MSE:  1   [6.457226989911679]
Local - R2:  1   [-17.986162447307173]
Training 1 clients in parallel...
Local - MSE:  1   [23.03077734728223]
Local - R2:  1   [-66.83526307337551]
Training 1 clients in parallel...
Local - MSE:  1   [